# Template · Discrete HMM (hmmlearn + Forward/Viterbi by hand)
Replace states, symbols, π, A, B and the observation sequence.

In [ ]:
%pip install -q hmmlearn

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from hmmlearn import hmm

In [ ]:
states = ["S1", "S2"]
symbols = ["a", "b", "c"]
pi = np.array([0.6, 0.4])
A = np.array([[0.7, 0.3],             # row = from, column = to
              [0.4, 0.6]])
B = np.array([[0.5, 0.4, 0.1],        # row = state, column = symbol
              [0.1, 0.3, 0.6]])
assert np.allclose(A.sum(1), 1) and np.allclose(B.sum(1), 1)

model = hmm.CategoricalHMM(n_components=len(states), init_params="", params="")
model.n_features = len(symbols)
model.startprob_, model.transmat_, model.emissionprob_ = pi, A, B

obs = ["a", "b", "c"]
X = np.array([symbols.index(s) for s in obs]).reshape(-1, 1)      # COLUMN of integers

In [ ]:
# Forward
alpha = pi * B[:, X[0, 0]]
for o in X[1:, 0]:
    alpha = B[:, o] * (alpha @ A)
print("P(obs) by hand  =", alpha.sum())
print("P(obs) hmmlearn =", np.exp(model.score(X)))

In [ ]:
# Viterbi + per-time posteriors
log_p, path = model.decode(X, algorithm="viterbi")
print("Viterbi:", [states[s] for s in path], "  P(path, obs) =", np.exp(log_p))
print("P(X_t | obs):\n", model.predict_proba(X))

In [ ]:
# Simulation: empirical probability of the sequence
N, hits = 10_000, 0
for k in range(N):
    o, _ = model.sample(len(obs), random_state=k)
    hits += np.array_equal(o.ravel(), X.ravel())
print("empirical:", hits / N, "  exact:", np.exp(model.score(X)))